# Pine Island Glacier: perturbed-ensemble projections to 2300 with pyISSM

This notebook:

1. **builds an initial state** of Pine Island Glacier (PIG): a 2D SSA model with a Budd-friction inversion
   against MEaSUREs velocities, BedMachine v3 geometry, RACMO SMB and AQ1 geothermal flux;
2. builds **idealised SMB and basal-melt (BMB) time series** for one **low** and one **high** emission scenario,
   smooth and with **no climate variability**;
3. generates **ensembles of perturbed initial states / forcings**: white noise (optionally smoothed) in the
   **grounded-ice melt rate**, the **bed topography** and/or the **basal friction coefficient**;
4. runs **prognostic simulations 2015–2300** for every member and scenario;
5. plots the ensemble spread in sea-level contribution and grounding-line position.

Throughout, cells marked **🔧 CHOICE** are decisions *you* need to make. Defaults are sensible starting points,
not recommendations. Cells marked **⚠️** flag modelling assumptions worth discussing.

**Requirements:** an NCI ARE JupyterLab session with the `access-issm` module loaded. Add
`gdata/au88+gdata/av17` to the session's *Storage* field. You need membership of `au88` (your work directory) and
`av17` (the Cryosphere Community Datapool), plus pyISSM and ccdtools installed somewhere under
`/g/data/au88/<username>`.

## 0. Paths and environment

**🔧 CHOICE:** where your pyISSM / ccdtools installs live, the project root, and how many processors each ISSM
run gets. Everything is written under `/g/data/au88/<username>/...`.

`execution_path` must **not** be the folder this notebook is in.

In [ ]:
import os, sys, glob, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

USER = os.environ['USER']                       # your NCI username

# 🔧 CHOICE: where you cloned/installed pyISSM and ccdtools
sys.path.append(f'/g/data/au88/{USER}/python_packages/pyISSM/src')
sys.path.append(f'/g/data/au88/{USER}/python_packages/ccdtools/src')
import pyissm
import ccdtools

# 🔧 CHOICE: project root and run settings
root_path      = f'/g/data/au88/{USER}/pig_ensemble'
model_path     = f'{root_path}/models'       # initial state (+ optional full transient models)
results_path   = f'{root_path}/results'      # per-run time series (csv) and grounding lines (npz)
plot_path      = f'{root_path}/plots'
data_path      = f'{root_path}/data'         # put pig_outline.exp here
execution_path = f'{root_path}/execution'    # ISSM scratch; must differ from this notebook's folder

for p in [model_path, results_path, plot_path, data_path, execution_path]:
    os.makedirs(p, exist_ok=True)

n_procs = 4        # 🔧 CHOICE: processors per ISSM run (check against your ARE request)

init_model_file = f'{model_path}/pig_ssa_init.nc'
print('pyISSM', getattr(pyissm, '__version__', '?'), '| root:', root_path)

## 1. Initial state

This is a condensed version of the standard PIG set-up. It covers the mesh, masks, geometry with hydrostatic
equilibrium on the shelf, temperature-dependent rheology, RACMO SMB, AQ1 geothermal flux, Budd friction
(p = q = 1), marine ice-sheet boundary conditions, SSA, and the inversion for the friction coefficient.

The whole build runs only if `init_model_file` doesn't exist yet, or if you force it. It saves a single
netCDF file that every ensemble member starts from.

**🔧 CHOICE:** mesh resolution, the year used for SMB and temperature, how far to lower the sub-shelf bathymetry,
and the inversion cost weights and bounds. **⚠️** Every ensemble member inherits these choices.

In [ ]:
# 🔧 CHOICE: initial-state settings
FORCE_REBUILD_INIT = False                 # True = rebuild even if init_model_file exists
domain_file = f'{data_path}/pig_outline.exp'

MESH = dict(hinit=10e3, hmax=40e3, hmin=5e3, gradation=1.7, err=8)   # bamg settings (m, m/yr)
CLIMATOLOGY_YEAR = 1995                    # RACMO year for SMB and skin temperature
SHELF_BATHY_OFFSET = 1000.                 # m; lowers the poorly known sub-shelf bathymetry (0 to switch off)
INVERSION = dict(cost_functions=[101, 103, 501], weights=[50, 1, 1e-19], cmin=1., cmax=200.)

In [ ]:
def build_initial_state(save_to):
    # Build, invert and save the PIG SSA initial state.
    catalog = ccdtools.catalog.DataCatalog()
    bm  = catalog.load_dataset('measures_bedmachine_antarctica', version='v3')
    vel = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version='v2')
    rac = catalog.load_dataset('racmo2.3p2_monthly_27km_1979-2022')
    aq1 = catalog.load_dataset('antarctic_geothermal_heat_flow_model_aq1', resolution='20km')
    interp = pyissm.data.interp

    # --- mesh: uniform, then refined on observed speed ---------------------
    md = pyissm.model.mesh.bamg(pyissm.model.Model(), domain=domain_file, hmax=MESH['hinit'])
    speed = np.hypot(interp.xr_to_mesh(vel, 'VX', md.mesh.x, md.mesh.y, fill_nan=True),
                     interp.xr_to_mesh(vel, 'VY', md.mesh.x, md.mesh.y, fill_nan=True))
    md = pyissm.model.mesh.bamg(md, hmax=MESH['hmax'], hmin=MESH['hmin'], gradation=MESH['gradation'],
                                field=speed, err=MESH['err'])
    md.miscellaneous.name = 'PIG_init'
    md.mesh.epsg = 3031
    print(f'mesh: {md.mesh.numberofvertices} vertices, {md.mesh.numberofelements} elements')

    # --- level sets from the BedMachine mask (0 ocean, 1 land, 2 grounded, 3 floating, 4 lake) ----
    mask = interp.xr_to_mesh(bm, 'mask', md.mesh.x, md.mesh.y, interpolation_type='nearest')
    md.mask.ice_levelset = np.where(mask > 0, -1., 1.)
    md.mask.ocean_levelset = np.where((mask == 0) | (mask == 3), -1., 1.)

    # --- geometry, with hydrostatic equilibrium on floating ice ------------
    md.geometry.base = interp.xr_to_mesh(bm, 'bed', md.mesh.x, md.mesh.y)
    md.geometry.surface = interp.xr_to_mesh(bm, 'surface', md.mesh.x, md.mesh.y)
    md.geometry.thickness = md.geometry.surface - md.geometry.base
    di = md.materials.rho_ice / md.materials.rho_water
    fl = md.mask.ocean_levelset < 0
    md.geometry.thickness[fl] = md.geometry.surface[fl] / (1 - di)
    md.geometry.base[fl] = md.geometry.surface[fl] - md.geometry.thickness[fl]
    md.geometry.thickness = np.maximum(md.geometry.thickness, 1.)
    md.geometry.surface = md.geometry.base + md.geometry.thickness
    md.geometry.bed = md.geometry.base.copy()
    md.geometry.bed[fl] -= SHELF_BATHY_OFFSET

    # --- temperature, rheology, velocity -----------------------------------
    xr_, yr_ = pyissm.tools.general.ll_to_xy(rac.lat, rac.lon, -1)
    sel = rac.sel(time=rac['time.year'] == CLIMATOLOGY_YEAR)
    tskin = sel['tskin'].mean('time').squeeze().values
    md.initialization.temperature = np.minimum(interp.points_to_mesh(xr_, yr_, tskin, md.mesh.x, md.mesh.y), 273.15)
    md.materials.rheology_n = 3 * np.ones(md.mesh.numberofelements)
    md.materials.rheology_B = pyissm.tools.materials.paterson(md.initialization.temperature)
    md.initialization.vx = interp.xr_to_mesh(vel, 'VX', md.mesh.x, md.mesh.y, fill_nan=True)
    md.initialization.vy = interp.xr_to_mesh(vel, 'VY', md.mesh.x, md.mesh.y, fill_nan=True)
    md.initialization.vz = np.zeros(md.mesh.numberofvertices)
    md.initialization.pressure = md.materials.rho_ice * md.constants.g * md.geometry.thickness
    md.inversion.vx_obs = md.initialization.vx.copy()
    md.inversion.vy_obs = md.initialization.vy.copy()
    md.inversion.vel_obs = np.hypot(md.inversion.vx_obs, md.inversion.vy_obs)

    # --- SMB (m ice eq./yr) and geothermal flux (W/m2) ----------------------
    smb = sel['smb'].groupby('time.year').sum('time').squeeze().values / md.materials.rho_ice
    md.smb.mass_balance = interp.points_to_mesh(xr_, yr_, smb, md.mesh.x, md.mesh.y)
    md.basalforcings.geothermalflux = interp.xr_to_mesh(aq1, 'Q', md.mesh.x, md.mesh.y, x_var='X', y_var='Y',
                                                        crop_buffer=20000, fill_nan=True)

    # --- friction (Budd, p = q = 1), BCs, SSA -------------------------------
    md.friction.coefficient = 10. * np.ones(md.mesh.numberofvertices)
    md.friction.coefficient[fl] = 0.
    md.friction.p = np.ones(md.mesh.numberofelements)
    md.friction.q = np.ones(md.mesh.numberofelements)
    md.groundingline.migration = 'SubelementMigration'
    md = pyissm.model.bc.set_marine_ice_sheet_bc(md)
    md.basalforcings.floatingice_melting_rate = np.zeros(md.mesh.numberofvertices)
    md.basalforcings.groundedice_melting_rate = np.zeros(md.mesh.numberofvertices)
    md.thermal.spctemperature = md.initialization.temperature.copy()
    md.masstransport.spcthickness = np.full(md.mesh.numberofvertices, np.nan)
    md = pyissm.model.param.set_flow_equation(md, SSA='all')

    # --- inversion for the friction coefficient ------------------------------
    md.inversion.iscontrol = 1
    md.inversion.cost_functions = INVERSION['cost_functions']
    md.inversion.cost_functions_coefficients = np.ones((md.mesh.numberofvertices, len(INVERSION['weights'])))
    for k, w in enumerate(INVERSION['weights']):
        md.inversion.cost_functions_coefficients[:, k] = w
    md.inversion.control_parameters = ['FrictionCoefficient']
    md.inversion.min_parameters = INVERSION['cmin'] * np.ones(md.mesh.numberofvertices)
    md.inversion.max_parameters = INVERSION['cmax'] * np.ones(md.mesh.numberofvertices)
    md.verbose = md.verbose.deactivate_all()
    md.verbose.control = 1
    md.cluster.np = n_procs
    md.cluster.executionpath = execution_path
    md = pyissm.model.execute.solve(md, 'Stressbalance')

    # Copy the inverted C into the model and switch the inversion off (easy to forget!)
    md.friction.coefficient = md.results.StressbalanceSolution.FrictionCoefficient.copy()
    md.inversion.iscontrol = 0
    pyissm.model.io.save_model(md, save_to)
    print('saved', save_to)
    return md


if FORCE_REBUILD_INIT or not os.path.exists(init_model_file):
    md_init = build_initial_state(init_model_file)
else:
    print('using existing initial state:', init_model_file)

In [ ]:
def load_initial_model(name, n_procs=n_procs):
    # Fresh copy of the initial state, ready for a forward run.
    md = pyissm.model.io.load_model(init_model_file)
    md.miscellaneous.name = name
    md.mesh.elements = md.mesh.elements.astype(int)
    md.cluster = pyissm.model.classes.cluster.generic()   # the saved model remembers the old node
    md.cluster.np = n_procs
    md.cluster.executionpath = execution_path
    md.inversion.iscontrol = 0
    md.verbose = md.verbose.deactivate_all()
    return md


md_init = load_initial_model('PIG_init')
node_types = pyissm.model.mesh.find_node_types(md_init, md_init.mask.ice_levelset, md_init.mask.ocean_levelset)
grounded0 = node_types['grounded_ice_nodes']
floating0 = node_types['floating_ice_nodes']

fig, axs = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
pyissm.plot.plot_model_field(md_init, md_init.geometry.bed, ax=axs[0], cmap='terrain', vmin=-1500, vmax=1500,
                             show_cbar=True, cbar_kwargs={'label': 'Bed (m)'})
pyissm.plot.plot_model_field(md_init, md_init.friction.coefficient, ax=axs[1], ylabel='',
                             show_cbar=True, cbar_kwargs={'label': 'Friction coefficient C'})
pyissm.plot.plot_model_field(md_init, md_init.inversion.vel_obs, ax=axs[2], ylabel='', vmin=0, vmax=4000,
                             show_cbar=True, cbar_kwargs={'label': 'Observed speed (m/yr)'})
for ax, t in zip(axs, ['Bed', 'Inverted friction', 'Observed speed']):
    pyissm.plot.plot_contour(md_init, md_init.mask.ocean_levelset, levels=[0], ax=ax, colors='k', linewidths=0.8)
    ax.set_title(t); ax.set_aspect('equal')
plt.savefig(f'{plot_path}/initial_state.png', dpi=150)
plt.show()

## 2. Forcing: SMB and BMB time series (no variability)

Both forcings are smooth time series derived from an idealised warming pathway $\Delta T(t)$ (GSAT relative to
2015). There's no interannual or decadal variability, so all of the spread in Section 4 comes from the
ensemble perturbations.

* **SMB:** the present-day RACMO pattern scaled in time, $\mathrm{SMB}(x,t) = \mathrm{SMB}_0(x)\,[1 + \alpha_{smb}\,\Delta T(t)]$.
* **BMB:** ISSM's `basalforcings.linear` scheme. Melt varies linearly with ice-shelf draft between an upper
  depth (melt $m_{up}$) and a deep depth (melt $m_{deep}$). Because melt is evaluated on the *evolving* draft,
  it responds to shelf thinning and grounding-line retreat. The two melt rates are scalar **time series** scaled
  quadratically with the ocean thermal-forcing anomaly:

$$ m(t) = m_0 \left( \frac{TF_0 + \Delta TF(t)}{TF_0} \right)^2, \qquad
   \Delta TF(t) = \min\!\left(\beta\, \Delta T(t - \tau),\; \Delta TF_{max}\right) $$

**🔧 CHOICE:** the pathways, the sensitivities and the present-day melt profile. Calibrate $m_{deep,0}$ so the
present-day shelf-mean melt is plausible (PIG is about 16 m/yr; Rignot et al., 2013). The next cell prints your
value. If you have your own forcing (e.g. from a CMIP model), set `FORCING_CSV`.

In [ ]:
# 🔧 CHOICE: period and scenarios
START_YEAR, FINAL_YEAR = 2015, 2300
years = np.arange(START_YEAR, FINAL_YEAR + 1, dtype=float)   # annual forcing; ISSM interpolates in time

def gsat_low(y):    # roughly SSP1-2.6: about +1 K above 2015, then stabilises
    return 1.0 * (1 - np.exp(-(y - START_YEAR) / 30.))

def gsat_high(y):   # roughly SSP5-8.5 extended: about +4 K by 2100, +7 K by 2300
    t = y - START_YEAR
    return np.where(t <= 85, 0.048 * t, 0.048 * 85 + 0.0145 * (t - 85))

SCENARIOS = {'low': gsat_low, 'high': gsat_high}
RUN_CONTROL = True     # 🔧 CHOICE: dT = 0 run per member, to remove model drift (recommended)
if RUN_CONTROL:
    SCENARIOS = {'control': lambda y: 0. * y, **SCENARIOS}

# 🔧 CHOICE: sensitivities and the present-day melt profile
SMB_SENS = 0.05          # fractional SMB change per K
TF_SENS, TF_LAG, TF_MAX = 0.4, 30., 3.0   # K of TF per K GSAT, lag (yr), saturation (K)
TF0 = 2.0                # present-day thermal forcing (K) used to scale melt quadratically
MELT_PROFILE = dict(deepwater_elevation=-800., deepwater_melting_rate=50.,   # m, m/yr
                    upperwater_elevation=-400., upperwater_melting_rate=0.)

FORCING_CSV = None       # 🔧 CHOICE: optional csv with columns year, scenario, smb_anomaly_frac, melt_factor

In [ ]:
def scenario_forcing(scenario):
    # DataFrame of annual forcing for one scenario (year, dT, smb_anomaly_frac, dTF, melt_factor).
    if FORCING_CSV is not None:
        f = pd.read_csv(FORCING_CSV)
        f = f[f['scenario'] == scenario].sort_values('year')
        return f[(f['year'] >= START_YEAR) & (f['year'] <= FINAL_YEAR)].reset_index(drop=True)
    dT = SCENARIOS[scenario](years)
    dTF = np.minimum(TF_SENS * SCENARIOS[scenario](np.maximum(years - TF_LAG, START_YEAR)), TF_MAX)
    return pd.DataFrame({'year': years, 'dT': dT, 'smb_anomaly_frac': SMB_SENS * dT,
                         'dTF': dTF, 'melt_factor': ((TF0 + dTF) / TF0) ** 2})


def linear_melt(draft, p):
    # Melt rate from the linear draft-dependent profile (for checking the calibration).
    frac = np.clip((draft - p['upperwater_elevation']) / (p['deepwater_elevation'] - p['upperwater_elevation']), 0, 1)
    return p['upperwater_melting_rate'] + frac * (p['deepwater_melting_rate'] - p['upperwater_melting_rate'])


forcings = {s: scenario_forcing(s) for s in SCENARIOS}

# Present-day calibration check: area-weighted mean melt over the initial ice shelf
melt0 = np.where(floating0, linear_melt(md_init.geometry.base, MELT_PROFILE), np.nan)
print(f'present-day shelf-mean melt: {pyissm.tools.diagnostics.weighted_mean(md_init, melt0):.1f} m/yr '
      f'(compare with about 16 m/yr observed)')

colours = {'control': 'tab:grey', 'low': 'tab:blue', 'high': 'tab:red'}
fig, axs = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
for s, f in forcings.items():
    axs[0].plot(f['year'], f['dT'] if 'dT' in f else np.nan, color=colours.get(s), label=s)
    axs[1].plot(f['year'], 100 * f['smb_anomaly_frac'], color=colours.get(s))
    axs[2].plot(f['year'], f['melt_factor'] * MELT_PROFILE['deepwater_melting_rate'], color=colours.get(s))
for ax, lab in zip(axs, ['GSAT anomaly (K)', 'SMB anomaly (%)', 'Deep-water melt rate (m/yr)']):
    ax.set_ylabel(lab); ax.set_xlabel('Year'); ax.grid(alpha=0.3)
axs[0].legend()
plt.savefig(f'{plot_path}/forcing.png', dpi=150)
plt.show()

## 3. Ensemble design and perturbations

Each ensemble member perturbs one or more of:

| Perturbation | What changes | Noise |
|---|---|---|
| `grounded_melt` | `md.basalforcings.groundedice_melting_rate` (m/yr, positive = melt) | mean + σ·ε, redrawn every year (`time_varying=True`) or fixed |
| `bed` | bed under **grounded** ice; base and thickness updated with the surface held fixed | additive, σ in m |
| `friction` | Budd friction coefficient C under grounded ice | multiplicative log-normal, C·exp(σε − σ²/2) |

ε is spatially white noise on the mesh vertices. Optionally it's smoothed with `smooth_iters` passes of
neighbour averaging (`pyissm.tools.interp.averaging`) to give it a spatial correlation length of a few
elements, and then rescaled back to unit variance.

**The same random field is used for a given member in every scenario.** Member *k* in `low` and in `high`
differ only in their climate forcing, so differences between scenarios aren't contaminated by sampling noise.

**🔧 CHOICES:**
* `N_MEMBERS`: how many perturbed members per perturbation type. Think about cost (next cell) and whether
  the spread has converged (Section 5 has a check).
* `DESIGN`: `'one_at_a_time'` (a separate ensemble per perturbation type, for attributing spread) or
  `'combined'` (all selected perturbations together in every member).
* The noise amplitudes. Is σ = 50 m of bed noise realistic for PIG's BedMachine errors (see `errbed`)? Should
  grounded-ice melt be allowed to be negative (freeze-on)?

**⚠️** Perturbing bed or friction *after* the inversion means the initial state no longer matches the observed
velocities. Expect an initial adjustment ("shock") in the first years. Options: accept it and look at changes
relative to each member's control, re-invert each member (expensive), or add a relaxation spin-up.

In [ ]:
# 🔧 CHOICE: ensemble design
N_MEMBERS = 5                                       # perturbed members per perturbation type (or in total if combined)
PERTURBATIONS = ['grounded_melt', 'bed', 'friction']   # any subset
DESIGN = 'one_at_a_time'                            # 'one_at_a_time' or 'combined'
INCLUDE_UNPERTURBED = True                          # also run the unperturbed reference (member 0)
SEED = 2026

PERTURB_PARAMS = {
    'grounded_melt': dict(mean=0.0, sigma=0.5, smooth_iters=0, time_varying=True, allow_negative=False),
    'bed':           dict(sigma=50., smooth_iters=3),
    'friction':      dict(sigma=0.3, smooth_iters=3, cmin=1., cmax=200.),
}

# 🔧 CHOICE: run settings
TIME_STEP = 0.5            # yr; check the CFL condition against hmin and the max speed
OUTPUT_FREQUENCY = 10      # save every OUTPUT_FREQUENCY * TIME_STEP years
GL_SNAPSHOT_YEARS = [2100, 2200, 2300]   # grounding lines kept for mapping
SAVE_FULL_MODELS = False   # full transient netCDF per run: large! the csv/npz outputs are enough for the plots
MINUTES_PER_RUN = 15       # your own estimate, for the cost printout

In [ ]:
def smooth_noise(md, rng, smooth_iters=0):
    # Unit-variance spatial noise on the vertices, optionally smoothed by neighbour averaging.
    eps = rng.standard_normal(md.mesh.numberofvertices)
    if smooth_iters > 0:
        eps = np.asarray(pyissm.tools.interp.averaging(md, eps, smooth_iters)).ravel()
        eps = (eps - eps.mean()) / eps.std()
    return eps


def perturb_grounded_melt(md, rng, years, p):
    # White-noise grounded-ice melt rate: a static field or an annual (nv+1, nt) time series.
    nv = md.mesh.numberofvertices
    if p['time_varying']:
        field = np.column_stack([p['mean'] + p['sigma'] * smooth_noise(md, rng, p['smooth_iters']) for _ in years])
    else:
        field = p['mean'] + p['sigma'] * smooth_noise(md, rng, p['smooth_iters'])
    if not p['allow_negative']:
        field = np.maximum(field, 0.)
    if p['time_varying']:
        field = np.vstack([field, years[None, :]])
    md.basalforcings.groundedice_melting_rate = field
    return md


def perturb_bed(md, rng, p):
    # Additive bed noise under grounded ice: surface held fixed, thickness and base follow.
    # Nodes that become floatable are re-set to hydrostatic equilibrium and flagged as floating.
    g = md.mask.ocean_levelset > 0
    dz = p['sigma'] * smooth_noise(md, rng, p['smooth_iters'])
    bed = md.geometry.bed.copy()
    bed[g] += dz[g]
    di = md.materials.rho_ice / md.materials.rho_water
    s = md.geometry.surface.copy()
    H = np.where(g, np.maximum(s - bed, 1.), md.geometry.thickness)
    base = np.where(g, s - H, md.geometry.base)
    # Flotation check: grounded nodes that are now too thin to stay grounded
    now_floats = g & (H * di < -bed)
    H[now_floats] = s[now_floats] / (1 - di)
    base[now_floats] = s[now_floats] - H[now_floats]
    bed[now_floats] = np.minimum(bed[now_floats], base[now_floats])
    md.mask.ocean_levelset = np.where(now_floats, -1., md.mask.ocean_levelset)
    md.friction.coefficient[now_floats] = 0.
    md.geometry.bed, md.geometry.base, md.geometry.thickness = bed, base, H
    md.geometry.surface = base + H
    md.initialization.pressure = md.materials.rho_ice * md.constants.g * H
    return md, int(now_floats.sum())


def perturb_friction(md, rng, p):
    # Multiplicative log-normal noise on C under grounded ice (mean-preserving), clipped to bounds.
    g = md.mask.ocean_levelset > 0
    eps = smooth_noise(md, rng, p['smooth_iters'])
    C = md.friction.coefficient.copy()
    C[g] = np.clip(C[g] * np.exp(p['sigma'] * eps[g] - 0.5 * p['sigma'] ** 2), p['cmin'], p['cmax'])
    md.friction.coefficient = C
    return md


def apply_perturbations(md, ptypes, member, years):
    # Apply the perturbations for one member, using reproducible per-member, per-type random streams.
    info = {}
    for ptype in ptypes:
        rng = np.random.default_rng([SEED, member, PERTURBATIONS.index(ptype)])
        p = PERTURB_PARAMS[ptype]
        if ptype == 'grounded_melt':
            md = perturb_grounded_melt(md, rng, years, p)
        elif ptype == 'bed':
            md, info['bed_nodes_now_floating'] = perturb_bed(md, rng, p)
        elif ptype == 'friction':
            md = perturb_friction(md, rng, p)
    return md, info


def make_plan():
    # One row per run: scenario x perturbation group x member.
    groups = [[p] for p in PERTURBATIONS] if DESIGN == 'one_at_a_time' else [PERTURBATIONS]
    rows = []
    for scenario in SCENARIOS:
        if INCLUDE_UNPERTURBED:
            rows.append(dict(scenario=scenario, group='none', ptypes=[], member=0))
        for grp in groups:
            gname = '+'.join(grp)
            for m in range(1, N_MEMBERS + 1):
                rows.append(dict(scenario=scenario, group=gname, ptypes=grp, member=m))
    plan = pd.DataFrame(rows)
    plan['name'] = [f"PIG_{r.scenario}_{r.group.replace('+', '-')}_m{r.member:03d}" for r in plan.itertuples()]
    return plan


plan = make_plan()
print(f'{len(plan)} runs  (about {len(plan) * MINUTES_PER_RUN / 60:.1f} h at {MINUTES_PER_RUN} min each, '
      f'on {n_procs} cpus per run)')
plan.groupby(['scenario', 'group']).size().unstack(0)

In [ ]:
# Preview: what member 1 of each perturbation type looks like (differences from the initial state)
fig, axs = plt.subplots(1, len(PERTURBATIONS), figsize=(6 * len(PERTURBATIONS), 5), constrained_layout=True,
                        squeeze=False)
for ax, ptype in zip(axs[0], PERTURBATIONS):
    md_p, info = apply_perturbations(load_initial_model('preview'), [ptype], 1, years)
    if ptype == 'bed':
        field, lab = md_p.geometry.bed - md_init.geometry.bed, 'Bed perturbation (m)'
        print(f"bed: {info['bed_nodes_now_floating']} grounded nodes became floating")
    elif ptype == 'friction':
        field = np.where(grounded0, md_p.friction.coefficient / np.maximum(md_init.friction.coefficient, 1e-9), np.nan)
        lab = 'C perturbed / C initial'
    else:
        gm = md_p.basalforcings.groundedice_melting_rate
        field, lab = (gm[:-1, 0] if gm.ndim == 2 else gm), 'Grounded melt, first year (m/yr)'
    if ptype == 'bed':
        lim = np.nanmax(np.abs(field))
        kw = dict(cmap='RdBu_r', vmin=-lim, vmax=lim)
    elif ptype == 'friction':
        kw = dict(cmap='PuOr', vmin=0, vmax=2)
    else:
        field = np.where(grounded0, field, np.nan)          # grounded melt only acts on grounded ice
        kw = dict(cmap='viridis')
    pyissm.plot.plot_model_field(md_init, field, ax=ax, show_cbar=True, cbar_kwargs={'label': lab}, **kw)
    pyissm.plot.plot_contour(md_init, md_init.mask.ocean_levelset, levels=[0], ax=ax, colors='k', linewidths=0.8)
    ax.set_title(ptype); ax.set_aspect('equal')
plt.savefig(f'{plot_path}/perturbation_preview.png', dpi=150)
plt.show()

## 4. Prognostic runs

Each run loads a fresh copy of the initial state, applies the member's perturbations, sets the scenario
forcing and solves a transient (stress balance + mass transport + grounding line; no thermal; fixed calving
front) from 2015 to 2300.

Per run the notebook saves a small **csv time series** and the **grounding line (ocean level set)** at
`GL_SNAPSHOT_YEARS`. The loop **skips runs that already have a csv**, so you can stop and restart it.

**🔧 CHOICE:**
* `QUICK_TEST = True` first. It runs 2 short runs to 2025 to check everything works. Then set it to `False`.
* **Run everything here, or batch it?** For more than about 10 runs, put the loop in a `.py` file and submit it
  with `qsub`. Gadi has no PBS job arrays, so use one job per scenario or group, or `nci-parallel`. The functions
  in this notebook import cleanly into a script.

**⚠️** The calving front is fixed (`ismovingfront = 0`), the ice hardness is fixed, and the BMB pattern only
evolves through the draft. Keep these in mind when interpreting results to 2300.

In [ ]:
def set_forcing(md, f):
    # SMB as an (nv+1, nt) series; BMB via the linear draft-dependent scheme with time-varying melt rates.
    yrs = f['year'].values.astype(float)
    smb0 = np.asarray(md.smb.mass_balance, dtype=float)
    smb0 = smb0[:-1, 0] if smb0.ndim == 2 else smb0
    md.smb.mass_balance = np.vstack([smb0[:, None] * (1 + f['smb_anomaly_frac'].values[None, :]), yrs[None, :]])

    gmelt = md.basalforcings.groundedice_melting_rate      # keep any perturbation already applied
    geo = md.basalforcings.geothermalflux
    md.basalforcings = pyissm.model.classes.basalforcings.linear()
    md.basalforcings.groundedice_melting_rate = gmelt
    md.basalforcings.geothermalflux = geo
    md.basalforcings.deepwater_elevation = MELT_PROFILE['deepwater_elevation']
    md.basalforcings.upperwater_elevation = MELT_PROFILE['upperwater_elevation']
    md.basalforcings.deepwater_melting_rate = np.vstack([MELT_PROFILE['deepwater_melting_rate'] * f['melt_factor'].values, yrs])
    md.basalforcings.upperwater_melting_rate = np.vstack([MELT_PROFILE['upperwater_melting_rate'] * f['melt_factor'].values, yrs])
    return md


def setup_transient(md, final_year):
    md.transient.isstressbalance = 1
    md.transient.ismasstransport = 1
    md.transient.isgroundingline = 1
    md.transient.issmb = 0
    md.transient.isthermal = 0
    md.transient.ismovingfront = 0
    md.groundingline.migration = 'SubelementMigration'
    md.groundingline.friction_interpolation = 'SubelementFriction1'
    md.groundingline.melt_interpolation = 'SubelementMelt1'   # 🔧 CHOICE: sub-element melt scheme
    md.masstransport.stabilization = 1
    md.masstransport.min_thickness = 10.
    md.masstransport.spcthickness = np.full(md.mesh.numberofvertices, np.nan)
    md.timestepping.start_time = START_YEAR
    md.timestepping.final_time = final_year
    md.timestepping.time_step = TIME_STEP
    md.timestepping.interp_forcing = 1
    md.timestepping.cycle_forcing = 0
    md.settings.output_frequency = OUTPUT_FREQUENCY
    md.transient.requested_outputs = ['default', 'IceVolume', 'IceVolumeAboveFloatation', 'GroundedArea',
                                      'FloatingArea', 'TotalSmb', 'TotalFloatingBmb', 'TotalGroundedBmb',
                                      'GroundinglineMassFlux', 'MaskOceanLevelset']
    return md


def build_run(row, final_year=FINAL_YEAR):
    md = load_initial_model(row['name'])
    md, info = apply_perturbations(md, row['ptypes'], row['member'], years)
    md = set_forcing(md, forcings[row['scenario']])
    md = setup_transient(md, final_year)
    return md, info


def vaf_to_slc(vaf, vaf_ref, rho_ice=917., rho_water=1027., ocean_area=3.625e14):
    # Sea-level contribution (mm) from a change in volume above flotation.
    return -1e3 * (np.asarray(vaf, float) - vaf_ref) * (rho_ice / rho_water) / ocean_area


def extract_timeseries(md):
    r = md.results.TransientSolution
    vaf = np.asarray(r.IceVolumeAboveFloatation, float)
    return pd.DataFrame({
        'year': np.asarray(r.time, float),
        'vaf_m3': vaf,
        'slc_mm': vaf_to_slc(vaf, vaf[0], md.materials.rho_ice, md.materials.rho_water),  # rel. to own first output
        'grounded_area_km2': np.asarray(r.GroundedArea, float) / 1e6,
        'floating_area_km2': np.asarray(r.FloatingArea, float) / 1e6,
        'total_smb_Gt_yr': np.asarray(r.TotalSmb, float),
        'total_floating_bmb_Gt_yr': np.asarray(r.TotalFloatingBmb, float),
        'total_grounded_bmb_Gt_yr': np.asarray(r.TotalGroundedBmb, float),
        'gl_flux_Gt_yr': np.asarray(r.GroundinglineMassFlux, float),
    })


def run_ensemble(plan, final_year=FINAL_YEAR, overwrite=False):
    log = []
    for i, (_, row) in enumerate(plan.iterrows()):
        csv = f"{results_path}/{row['name']}.csv"
        if os.path.exists(csv) and not overwrite:
            continue
        t0 = time.time()
        try:
            md, info = build_run(row, final_year)
            md = pyissm.model.execute.solve(md, 'Transient')
            ts = extract_timeseries(md)
            ts.assign(scenario=row['scenario'], group=row['group'], member=row['member']).to_csv(csv, index=False)
            r = md.results.TransientSolution
            t = np.asarray(r.time, float)
            gl = {str(y): np.asarray(r.MaskOceanLevelset[int(np.argmin(np.abs(t - y)))], float)
                  for y in GL_SNAPSHOT_YEARS if y <= t.max() + 1e-6}
            np.savez_compressed(f"{results_path}/{row['name']}_gl.npz", **gl)
            if SAVE_FULL_MODELS:
                pyissm.model.io.save_model(md, f"{model_path}/{row['name']}.nc")
            status = 'ok'
        except Exception as e:                       # keep going; inspect failures afterwards
            status, info = f'FAILED: {e}', {}
        log.append(dict(name=row['name'], status=status, minutes=(time.time() - t0) / 60, **info))
        print(f"[{i + 1}/{len(plan)}] {row['name']}: {status} ({log[-1]['minutes']:.1f} min)")
    return pd.DataFrame(log)

In [ ]:
# 🔧 CHOICE: test first, then run the full ensemble
QUICK_TEST = True

if QUICK_TEST:
    test_plan = plan.groupby('scenario').head(1).head(2).copy()
    test_plan['name'] = test_plan['name'] + '_TEST'
    run_log = run_ensemble(test_plan, final_year=START_YEAR + 10, overwrite=True)
else:
    run_log = run_ensemble(plan)
run_log

## 5. Analysis and plots

All plots read the per-run csv files, so they work after a kernel restart without re-running anything.

**🔧 CHOICE:** `RELATIVE_TO_CONTROL = True` shows the *forced* response: each member minus the control run of the
**same member**, which removes both model drift and the perturbation's own initial adjustment. `False` shows
absolute changes.

In [ ]:
RELATIVE_TO_CONTROL = RUN_CONTROL      # 🔧 CHOICE

files = [f for f in sorted(glob.glob(f'{results_path}/PIG_*.csv')) if not f.endswith('_TEST.csv')]
ts = pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()
print(f'{len(files)} runs found')

if RELATIVE_TO_CONTROL and len(ts):
    ctrl = ts[ts['scenario'] == 'control'][['group', 'member', 'year', 'slc_mm', 'grounded_area_km2']]
    ts = ts.merge(ctrl, on=['group', 'member', 'year'], suffixes=('', '_ctrl'), how='left')
    ts['slc_plot'] = ts['slc_mm'] - ts['slc_mm_ctrl']
    ts['ga_plot'] = ts['grounded_area_km2'] - ts['grounded_area_km2_ctrl']
    ts = ts[ts['scenario'] != 'control']
    ylab = 'SLC relative to control (mm)'
else:
    ts['slc_plot'] = ts.get('slc_mm')
    ts['ga_plot'] = ts.get('grounded_area_km2') - ts.groupby(['scenario', 'group', 'member'])['grounded_area_km2'].transform('first') if len(ts) else None
    ylab = 'Sea-level contribution (mm)'

In [ ]:
# 1) SLC through time: one panel per perturbation group, members thin, ensemble mean thick
groups = [g for g in ts['group'].unique() if g != 'none']
fig, axs = plt.subplots(1, max(len(groups), 1), figsize=(6 * max(len(groups), 1), 4.5), sharey=True,
                        constrained_layout=True, squeeze=False)
for ax, grp in zip(axs[0], groups):
    for scen, d in ts[ts['group'] == grp].groupby('scenario'):
        for _, dm in d.groupby('member'):
            ax.plot(dm['year'], dm['slc_plot'], color=colours[scen], lw=0.6, alpha=0.5)
        mean = d.groupby('year')['slc_plot'].mean()
        ax.plot(mean.index, mean.values, color=colours[scen], lw=2.2, label=f'{scen} (mean)')
    ref = ts[ts['group'] == 'none']
    for scen, d in ref.groupby('scenario'):
        ax.plot(d['year'], d['slc_plot'], color=colours[scen], lw=1.5, ls='--')
    ax.set_title(f'Perturbation: {grp}'); ax.set_xlabel('Year'); ax.grid(alpha=0.3)
axs[0][0].set_ylabel(ylab)
handles = [Line2D([], [], color=colours[s_], lw=2.2, label=f'{s_} (ensemble mean)')
           for s_ in SCENARIOS if s_ in ts['scenario'].unique()]
handles += [Line2D([], [], color='k', lw=0.6, label='members'),
            Line2D([], [], color='k', lw=1.5, ls='--', label='unperturbed')]
axs[0][0].legend(handles=handles, fontsize=8, loc='upper left')
plt.savefig(f'{plot_path}/ensemble_slc_timeseries.png', dpi=150)
plt.show()

In [ ]:
# 2) Spread at 2100 and 2300: one dot per member, by scenario and perturbation group
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for ax, yr in zip(axs, [2100, 2300]):
    snap = ts.loc[ts.groupby(['scenario', 'group', 'member'])['year'].transform(lambda y: (y - yr).abs()).eq(
        ts.groupby(['scenario', 'group', 'member'])['year'].transform(lambda y: (y - yr).abs().min()))]
    cats = sorted(snap['group'].unique())
    for j, scen in enumerate([s for s in SCENARIOS if s in snap['scenario'].unique()]):
        for i, grp in enumerate(cats):
            v = snap[(snap['scenario'] == scen) & (snap['group'] == grp)]['slc_plot']
            x = i + (j - 0.5) * 0.3
            ax.scatter(np.full(len(v), x), v, color=colours[scen], s=18, alpha=0.8, label=scen if i == 0 else None)
            if len(v) > 1:
                ax.plot([x - 0.1, x + 0.1], [v.mean()] * 2, color='k', lw=1.5)
    ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats)
    ax.set_title(f'{yr}'); ax.set_ylabel(ylab); ax.grid(alpha=0.3, axis='y')
axs[0].legend(fontsize=8)
plt.savefig(f'{plot_path}/ensemble_spread.png', dpi=150)
plt.show()

summary = (ts[ts['year'].round().isin([2100, 2300])]
           .groupby(['scenario', 'group', ts['year'].round()])['slc_plot']
           .agg(['count', 'mean', 'std', 'min', 'max']).round(1))
summary

In [ ]:
# 3) Grounding lines in the final snapshot year for every member, over the bed
SNAP = str(GL_SNAPSHOT_YEARS[-1])
scen_list = [s for s in SCENARIOS if s != 'control']
fig, axs = plt.subplots(1, len(scen_list), figsize=(7 * len(scen_list), 6), constrained_layout=True, squeeze=False)
for ax, scen in zip(axs[0], scen_list):
    pyissm.plot.plot_model_field(md_init, md_init.geometry.bed, ax=ax, cmap='terrain', vmin=-1500, vmax=1500,
                                 show_cbar=True, cbar_kwargs={'label': 'Bed (m)'})
    pyissm.plot.plot_contour(md_init, md_init.mask.ocean_levelset, levels=[0], ax=ax, colors='k', linewidths=1.2)
    for k, grp in enumerate(groups):
        for f in sorted(glob.glob(f'{results_path}/PIG_{scen}_{grp.replace("+", "-")}_m*_gl.npz')):
            gl = np.load(f)
            if SNAP in gl:
                pyissm.plot.plot_contour(md_init, gl[SNAP], levels=[0], ax=ax, colors=f'C{k}', linewidths=0.7)
    ax.legend(handles=[Line2D([], [], color='k', label='2015')] +
                      [Line2D([], [], color=f'C{k}', label=f'{g} ({SNAP})') for k, g in enumerate(groups)],
              fontsize=8, loc='lower right')
    ax.set_title(f'{scen}: grounding lines in {SNAP}'); ax.set_aspect('equal')
plt.savefig(f'{plot_path}/ensemble_grounding_lines.png', dpi=150)
plt.show()

In [ ]:
# 4) Has the ensemble spread converged? Std of SLC at the final year vs number of members used
fig, ax = plt.subplots(figsize=(6, 4))
final = ts[ts['year'] == ts['year'].max()]
for (scen, grp), d in final[final['group'] != 'none'].groupby(['scenario', 'group']):
    v = d.sort_values('member')['slc_plot'].values
    n = np.arange(2, len(v) + 1)
    ax.plot(n, [v[:k].std(ddof=1) for k in n], marker='o', color=colours[scen],
            ls=['-', '--', ':'][groups.index(grp) % 3] if grp in groups else '-', label=f'{scen} / {grp}')
ax.set_xlabel('Number of members'); ax.set_ylabel(f'Std of SLC at {ts["year"].max():.0f} (mm)')
ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.savefig(f'{plot_path}/ensemble_convergence.png', dpi=150)
plt.show()

## 6. Things to discuss

* Is the spread from a given perturbation **larger or smaller than the low–high scenario difference**? When,
  and for which perturbation, does the scenario signal emerge from the ensemble spread?
* Which perturbation dominates the uncertainty by 2100, and by 2300? Does that change once the grounding line
  reaches the retrograde bed?
* How much of the spread is the initial **shock** from breaking the inversion (compare runs with and without
  `RELATIVE_TO_CONTROL`)? Would re-inverting each member change your conclusions?
* Has the spread converged with `N_MEMBERS` (plot 4)? What's the smallest ensemble you'd trust?
* What would change with time-correlated grounded melt, spatially correlated bed errors scaled by BedMachine
  `errbed`, or a moving calving front?

### References
Morlighem et al. (2020) *Nat. Geosci.* 13, 132–137 · Rignot et al. (2017) MEaSUREs InSAR velocity v2, NSIDC ·
van Wessem et al. (2018) *TC* 12, 1479–1498 · Stål et al. (2021) *G-Cubed* 22 · Rignot et al. (2013) *Science*
341, 266–270 · Seroussi et al. (2020) *TC* 14, 3033–3070